# Tier 0 Assignment 1: Computational Foundations

**Covers**: Unix shell pipelines, Git, Bash scripting, file encodings, R-style statistics, chi-square, binomial probability, and Welch's t-test

---

## Grading Rubric

| Problem | Points | Difficulty |
|---------|--------|------------|
| 1. Shell Pipeline Simulator | 10 | (1 star) |
| 2. Git Log Parser | 10 | (1 star) |
| 3. Bash Variable Expander | 10 | (2 stars) |
| 4. File Encoding Detector | 15 | (2 stars) |
| 5. R Vector Statistics | 15 | (2 stars) |
| 6. Chi-Square Test | 15 | (2 stars) |
| 7. Binomial Probability | 15 | (3 stars) |
| 8. Independent T-Test (Welch's) | 10 | (3 stars) |
| **Total** | **100** | |

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook teaches core computational literacy used before any serious bioinformatics analysis: reproducible shell workflows, stats reasoning, and environment hygiene.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Difference between command syntax and conceptual model (what changes state vs what only inspects).
- Interpreting statistical output: p-value alone is not enough without effect size and assumptions.
- Reproducibility: the same command can yield different outputs if input files or working directory differ.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


In [ ]:
import math
import re

---

## Problem 1: Shell Pipeline Simulator (1 star)

Simulate a Unix pipeline on a list of strings:

```
grep <pattern> | cut -d' ' -f<field> | sort | uniq -c | sort -rn | head -n <top_n>
```

Steps:
1. **grep**: keep only lines that contain `pattern` as a substring
2. **cut**: split each surviving line on whitespace and take field at index `field` (0-based)
3. **sort | uniq -c**: count occurrences of each unique field value
4. **sort -rn | head -n**: return the top-N entries sorted by count descending

**Example**:
```python
lines = [
    "error disk full",
    "error network timeout",
    "warning disk space",
    "error disk corrupt",
]
shell_pipeline(lines, pattern="error", field=1, top_n=2)
# Returns: [(3, 'disk'), (1, 'network')]  — wait, 'disk' appears in 2 error lines
# Returns: [(2, 'disk'), (1, 'network')]
```

**Grading**: 10 points — 2 pts grep filter, 2 pts cut field, 3 pts count logic, 3 pts sort/top-N

In [ ]:
def shell_pipeline(lines: list, pattern: str, field: int, top_n: int) -> list:
    """
    Simulate a Unix grep | cut | sort | uniq -c | sort -rn | head pipeline.

    Args:
        lines:   List of text strings to process.
        pattern: Substring to filter lines by (grep step).
        field:   0-based index of the whitespace-separated field to extract (cut step).
        top_n:   Number of top results to return.

    Returns:
        List of (count, value) tuples sorted by count descending, limited to top_n.
        Lines that do not have enough fields after splitting are skipped.

    Example:
        >>> lines = ["error disk full", "error network timeout",
        ...          "warning disk space", "error disk corrupt"]
        >>> shell_pipeline(lines, pattern="error", field=1, top_n=2)
        [(2, 'disk'), (1, 'network')]
    """
    # YOUR CODE HERE
    pass


# Tests
lines = [
    "error disk full",
    "error network timeout",
    "warning disk space",
    "error disk corrupt",
]
print(shell_pipeline(lines, pattern="error", field=1, top_n=2))
# Expected: [(2, 'disk'), (1, 'network')]

log_lines = [
    "INFO server started",
    "ERROR server crashed",
    "ERROR server crashed",
    "INFO server started",
    "ERROR client disconnected",
]
print(shell_pipeline(log_lines, pattern="ERROR", field=1, top_n=3))
# Expected: [(2, 'server'), (1, 'client')]

---

## Problem 2: Git Log Parser (1 star)

Parse a git log string in the following format:

```
commit <full_hash>
Date:   <date string>

    <commit message>

commit <full_hash>
...
```

Return a list of dicts with keys:
- `hash`: first 7 characters of the full commit hash
- `date`: the date string (stripped)
- `message`: the commit message (stripped)

**Example**:
```python
log = "commit abc1234def5678\nDate:   Mon Jan 1 12:00:00 2024\n\n    Initial commit\n"
parse_git_log(log)
# Returns: [{'hash': 'abc1234', 'date': 'Mon Jan 1 12:00:00 2024', 'message': 'Initial commit'}]
```

**Grading**: 10 points — 3 pts hash extraction (short), 3 pts date parsing, 4 pts message parsing and return format

In [ ]:
def parse_git_log(log_text: str) -> list:
    """
    Parse a git log string into a list of commit dicts.

    Args:
        log_text: Multi-line string in git log format:
                  'commit <hash>\nDate:   <date>\n\n    <message>\n'

    Returns:
        List of dicts, each with keys:
            hash    (str): Short 7-character commit hash.
            date    (str): Date string, stripped of leading/trailing whitespace.
            message (str): Commit message, stripped of leading/trailing whitespace.

    Example:
        >>> log = "commit abc1234def5678\nDate:   Mon Jan 1 2024\n\n    Add feature\n"
        >>> parse_git_log(log)
        [{'hash': 'abc1234', 'date': 'Mon Jan 1 2024', 'message': 'Add feature'}]
    """
    # YOUR CODE HERE
    pass


# Tests
sample_log = """commit a1b2c3d4e5f67890abcdef1234567890abcdef12
Date:   Mon Apr 10 09:00:00 2024 +0000

    Add genome alignment module

commit deadbeef0123456789abcdef0123456789abcdef
Date:   Sun Apr 9 18:30:00 2024 +0000

    Fix off-by-one in FASTA parser

commit cafebabe1234567890abcdef1234567890abcdef
Date:   Sat Apr 8 12:15:00 2024 +0000

    Initial commit
"""

commits = parse_git_log(sample_log)
for c in commits:
    print(c)
# Expected:
# {'hash': 'a1b2c3d', 'date': 'Mon Apr 10 09:00:00 2024 +0000', 'message': 'Add genome alignment module'}
# {'hash': 'deadbee', 'date': 'Sun Apr 9 18:30:00 2024 +0000', 'message': 'Fix off-by-one in FASTA parser'}
# {'hash': 'cafebab', 'date': 'Sat Apr 8 12:15:00 2024 +0000', 'message': 'Initial commit'}

---

## Problem 3: Bash Variable Expander (2 stars)

Implement a simple shell variable expander. Given a template string and a dict
of variable names to values, expand all variable references.

Support two forms:
- `${VAR}` — braced form (higher priority)
- `$VAR` — unbraced form (match longest valid identifier: `[A-Za-z_][A-Za-z0-9_]*`)

Variables not present in the dict expand to an empty string `""`.

**Example**:
```python
expand_vars("Hello $NAME, you are $AGE years old", {"NAME": "Alice", "AGE": "30"})
# Returns: "Hello Alice, you are 30 years old"

expand_vars("Path: ${HOME}/data", {"HOME": "/usr/local"})
# Returns: "Path: /usr/local/data"

expand_vars("$UNDEFINED is gone", {})
# Returns: " is gone"
```

**Grading**: 10 points — 3 pts braced form, 3 pts unbraced form, 4 pts missing variables expand to empty string

In [ ]:
def expand_vars(template: str, variables: dict) -> str:
    """
    Expand shell-style variable references in a template string.

    Args:
        template:  String containing $VAR or ${VAR} references.
        variables: Dict mapping variable names (str) to their string values.

    Returns:
        The template with all variable references replaced. Variables not
        present in `variables` are replaced with an empty string.

    Example:
        >>> expand_vars("Hello $NAME!", {"NAME": "World"})
        'Hello World!'
        >>> expand_vars("${A}${B}", {"A": "foo"})
        'foo'
    """
    # YOUR CODE HERE
    pass


# Tests
print(expand_vars("Hello $NAME, you are $AGE years old", {"NAME": "Alice", "AGE": "30"}))
# Expected: Hello Alice, you are 30 years old

print(expand_vars("Path: ${HOME}/data/${FILE}", {"HOME": "/usr/local", "FILE": "genome.fa"}))
# Expected: Path: /usr/local/data/genome.fa

print(expand_vars("$UNDEFINED is gone", {}))
# Expected:  is gone

print(expand_vars("${A}${B}_suffix", {"A": "chr", "B": "1"}))
# Expected: chr1_suffix

---

## Problem 4: File Encoding Detector (2 stars)

Given a byte string (`bytes`), detect its text encoding by inspecting the
byte order mark (BOM) or byte values. Return one of the following strings:

| Condition | Return |
|-----------|--------|
| Starts with `0xEF 0xBB 0xBF` | `"utf-8-bom"` |
| Starts with `0xFF 0xFE` | `"utf-16-le"` |
| Starts with `0xFE 0xFF` | `"utf-16-be"` |
| All bytes < 128 | `"ascii"` |
| Decodes as valid UTF-8 | `"utf-8"` |
| Otherwise | `"unknown"` |

Check conditions **in the order listed above**.

**Example**:
```python
detect_encoding(b"Hello world")           # "ascii"
detect_encoding(b"\xef\xbb\xbfHello")    # "utf-8-bom"
detect_encoding(b"\xff\xfeHello")        # "utf-16-le"
detect_encoding("Héllo".encode("utf-8")) # "utf-8"
detect_encoding(b"\x80\x81\x82")        # "unknown"
```

**Grading**: 15 points — 3 pts per correct case (5 cases)

In [ ]:
def detect_encoding(data: bytes) -> str:
    """
    Detect the text encoding of a byte string by inspecting BOMs and byte ranges.

    Args:
        data: The raw bytes to inspect.

    Returns:
        One of: 'utf-8-bom', 'utf-16-le', 'utf-16-be', 'ascii', 'utf-8', 'unknown'.
        Conditions are checked in that priority order.

    Example:
        >>> detect_encoding(b"Hello")
        'ascii'
        >>> detect_encoding(b"\xef\xbb\xbfHello")
        'utf-8-bom'
    """
    # YOUR CODE HERE
    pass


# Tests
print(detect_encoding(b"Hello world"))             # ascii
print(detect_encoding(b"\xef\xbb\xbfHello"))      # utf-8-bom
print(detect_encoding(b"\xff\xfeHello"))           # utf-16-le
print(detect_encoding(b"\xfe\xffHello"))           # utf-16-be
print(detect_encoding("Héllo".encode("utf-8")))    # utf-8
print(detect_encoding(b"\x80\x81\x82"))           # unknown

---

## Problem 5: R Vector Statistics (2 stars)

Implement R's `summary()` function for a numeric list in Python. Return a dict
with the following keys, all rounded to 2 decimal places:

- `min`: minimum value
- `Q1`: first quartile (25th percentile)
- `median`: 50th percentile
- `mean`: arithmetic mean
- `Q3`: third quartile (75th percentile)
- `max`: maximum value

Use **linear interpolation** for quantiles (R's default `type=7`):
- Sort the data.
- For quantile `q` over `n` values, compute virtual index `h = q * (n - 1)`
  (0-based).
- Result = `data[floor(h)] + (h - floor(h)) * (data[floor(h)+1] - data[floor(h)])`
- Edge case: if `h` is an integer, return `data[h]` directly.

Do **not** use `numpy`, `scipy`, or `statistics` module.

**Grading**: 15 points — 2 pts min/max, 3 pts mean, 5 pts Q1/median/Q3 interpolation, 5 pts match R output

In [ ]:
def r_summary(data: list) -> dict:
    """
    Compute R-style summary statistics for a numeric list.

    Args:
        data: Non-empty list of numeric values.

    Returns:
        Dict with keys: 'min', 'Q1', 'median', 'mean', 'Q3', 'max',
        all rounded to 2 decimal places.
        Quantiles use linear interpolation (R type=7):
        h = q * (n-1), result = data[floor(h)] + frac(h) * (data[ceil(h)] - data[floor(h)])

    Example:
        >>> r_summary([1, 2, 3, 4, 5])
        {'min': 1.0, 'Q1': 2.0, 'median': 3.0, 'mean': 3.0, 'Q3': 4.0, 'max': 5.0}
        >>> r_summary([4, 8, 15, 16, 23, 42])
        {'min': 4.0, 'Q1': 8.75, 'median': 15.5, 'mean': 18.0, 'Q3': 22.25, 'max': 42.0}
    """
    # YOUR CODE HERE
    pass


# Tests
print(r_summary([1, 2, 3, 4, 5]))
# Expected: {'min': 1.0, 'Q1': 2.0, 'median': 3.0, 'mean': 3.0, 'Q3': 4.0, 'max': 5.0}

print(r_summary([4, 8, 15, 16, 23, 42]))
# Expected: {'min': 4.0, 'Q1': 8.75, 'median': 15.5, 'mean': 18.0, 'Q3': 22.25, 'max': 42.0}

print(r_summary([10]))
# Expected: {'min': 10.0, 'Q1': 10.0, 'median': 10.0, 'mean': 10.0, 'Q3': 10.0, 'max': 10.0}

---

## Problem 6: Chi-Square Goodness-of-Fit Test (2 stars)

Implement a chi-square goodness-of-fit test **from scratch** (no scipy).

Given:
- `observed`: list of observed integer counts
- `expected_proportions`: list of floats that sum to 1.0

Compute:
- `E_i = expected_proportions[i] * sum(observed)`
- `χ² = Σ (O_i - E_i)² / E_i`
- `df = len(observed) - 1`

Return `{"chi2": float, "df": int}` with `chi2` rounded to 4 decimal places.

**Example**:
```python
chi_square_gof([50, 30, 20], [0.5, 0.3, 0.2])
# Returns: {'chi2': 0.0, 'df': 2}  (observed matches expected exactly)

chi_square_gof([60, 20, 20], [0.5, 0.3, 0.2])
# Returns: {'chi2': 6.6667, 'df': 2}
```

**Grading**: 15 points — 4 pts expected counts, 6 pts chi2 formula, 3 pts df, 2 pts return format

In [ ]:
def chi_square_gof(observed: list, expected_proportions: list) -> dict:
    """
    Compute a chi-square goodness-of-fit statistic.

    Args:
        observed:             List of observed integer counts.
        expected_proportions: List of floats summing to 1.0; same length as observed.

    Returns:
        Dict with keys:
            'chi2' (float): Chi-square statistic, rounded to 4 decimal places.
            'df'   (int):   Degrees of freedom = len(observed) - 1.

    Example:
        >>> chi_square_gof([50, 30, 20], [0.5, 0.3, 0.2])
        {'chi2': 0.0, 'df': 2}
        >>> chi_square_gof([60, 20, 20], [0.5, 0.3, 0.2])
        {'chi2': 6.6667, 'df': 2}
    """
    # YOUR CODE HERE
    pass


# Tests
print(chi_square_gof([50, 30, 20], [0.5, 0.3, 0.2]))
# Expected: {'chi2': 0.0, 'df': 2}

print(chi_square_gof([60, 20, 20], [0.5, 0.3, 0.2]))
# Expected: {'chi2': 6.6667, 'df': 2}

print(chi_square_gof([10, 10, 10, 10], [0.25, 0.25, 0.25, 0.25]))
# Expected: {'chi2': 0.0, 'df': 3}  (uniform, perfect fit)

---

## Problem 7: Binomial Probability (3 stars)

Implement the binomial probability mass function (PMF) and cumulative distribution
function (CDF) **from scratch** (no scipy, no numpy).

**PMF**: `P(X = k) = C(n, k) * p^k * (1-p)^(n-k)`

**CDF**: `P(X ≤ k) = Σ_{i=0}^{k} PMF(n, i, p)`

Use integer arithmetic for the combination `C(n, k)` to avoid floating-point
overflow:
```
C(n, k) = n! / (k! * (n-k)!)
```
Compute it iteratively (multiply/divide step by step to keep values as integers).

**Example**:
```python
binomial_pmf(10, 3, 0.5)   # ≈ 0.1172
binomial_cdf(10, 3, 0.5)   # ≈ 0.1719
```

**Grading**: 15 points — 4 pts combination (integer), 4 pts PMF, 4 pts CDF summation, 3 pts float return

In [ ]:
def binomial_pmf(n: int, k: int, p: float) -> float:
    """
    Compute the binomial probability mass function P(X = k).

    Args:
        n: Number of trials (non-negative integer).
        k: Number of successes (0 <= k <= n).
        p: Probability of success on each trial (0.0 <= p <= 1.0).

    Returns:
        P(X = k) as a float.

    Example:
        >>> round(binomial_pmf(10, 5, 0.5), 4)
        0.2461
    """
    # YOUR CODE HERE
    pass


def binomial_cdf(n: int, k: int, p: float) -> float:
    """
    Compute the binomial CDF P(X <= k).

    Args:
        n: Number of trials.
        k: Upper bound (inclusive).
        p: Probability of success.

    Returns:
        P(X <= k) as a float (sum of PMF from 0 to k).

    Example:
        >>> round(binomial_cdf(10, 5, 0.5), 4)
        0.623
    """
    # YOUR CODE HERE
    pass


# Tests
print(round(binomial_pmf(10, 3, 0.5), 4))   # Expected: 0.1172
print(round(binomial_cdf(10, 3, 0.5), 4))   # Expected: 0.1719
print(round(binomial_pmf(10, 5, 0.5), 4))   # Expected: 0.2461
print(round(binomial_cdf(10, 5, 0.5), 4))   # Expected: 0.623
print(round(binomial_pmf(20, 10, 0.3), 6))  # Expected: 0.030782
print(round(binomial_cdf(20, 10, 0.3), 6))  # Expected: 0.982791

---

## Problem 8: Independent T-Test (Welch's) (3 stars)

Implement an independent samples t-test using **Welch's formula** from scratch
(no scipy, no numpy).

Given two lists of floats `group1` and `group2`:

**T-statistic**:
```
t = (mean1 - mean2) / sqrt(var1/n1 + var2/n2)
```
where `var` is the **sample variance** (divide by n-1).

**Welch-Satterthwaite degrees of freedom**:
```
df = (var1/n1 + var2/n2)^2 / ( (var1/n1)^2/(n1-1) + (var2/n2)^2/(n2-1) )
```

Return `{"t_stat": float, "df": float}`, both rounded to 4 decimal places.

**Example**:
```python
welch_t_test([2.1, 2.5, 2.3, 2.7], [1.8, 1.6, 2.0, 1.9])
# Returns: {'t_stat': 3.6515, 'df': 5.9965}  (approximately)
```

**Grading**: 10 points — 2 pts mean, 2 pts sample variance, 3 pts t-statistic, 3 pts Welch df

In [ ]:
def welch_t_test(group1: list, group2: list) -> dict:
    """
    Perform an independent samples t-test using Welch's formula.

    Args:
        group1: List of floats (first sample, n >= 2).
        group2: List of floats (second sample, n >= 2).

    Returns:
        Dict with keys:
            't_stat' (float): Welch's t-statistic, rounded to 4 decimal places.
            'df'     (float): Welch-Satterthwaite degrees of freedom, rounded to 4 decimal places.

    Example:
        >>> welch_t_test([2.1, 2.5, 2.3, 2.7], [1.8, 1.6, 2.0, 1.9])
        {'t_stat': 3.6515, 'df': 5.9965}
    """
    # YOUR CODE HERE
    pass


# Tests
print(welch_t_test([2.1, 2.5, 2.3, 2.7], [1.8, 1.6, 2.0, 1.9]))
# Expected: {'t_stat': 3.6515, 'df': 5.9965}  (approximately)

print(welch_t_test([5.0, 5.0, 5.0], [5.0, 5.0, 5.0]))
# Expected: t_stat = 0.0 (identical groups)

print(welch_t_test([1, 2, 3, 4, 5], [6, 7, 8, 9, 10]))
# Expected: large negative t_stat, df close to 8.0

---

## Summary

In this assignment you practiced core computational foundations used throughout
bioinformatics:

1. **Shell Pipeline Simulator**: grep, cut, uniq, and sort logic in pure Python
2. **Git Log Parser**: regex-based parsing of structured text output
3. **Bash Variable Expander**: regex substitution with priority rules
4. **File Encoding Detector**: byte-level inspection and BOM detection
5. **R Vector Statistics**: quantile interpolation and summary statistics
6. **Chi-Square Test**: goodness-of-fit hypothesis testing from scratch
7. **Binomial Probability**: PMF and CDF with exact integer arithmetic
8. **Welch's T-Test**: two-sample testing with Satterthwaite degrees of freedom
